In [20]:
import pandas as pd
import os 
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

In [21]:
filepath = Path("../Dataset/dataset_musique_region.csv")
if not filepath.exists():
    raise FileNotFoundError(f"Fichier introuvable : {filepath.resolve()}. Exécuter d’abord le notebook de nettoyage.")
df = pd.read_csv(filepath)
colonnes_genres = df.columns.drop("region")

# Transformation des données

préparer et comparer des variables  pour la prédiction géographique à partir des préférences musicales

- On veut trouver les régions les plus petites



# Transformation des données

L'objectif de cette étape est de préparer le dataset pour faciliter l'analyse des préférences musicales et étudier leur potentiel pour prédire la localisation géographique des individus.


Plusieurs transformations sont réalisées afin de :
- Etudier la répartition des individus selon leur région
- Réduire le nombre de catégories géographiques en regroupant certaines régions
- Analyser la fréquence de séléection des différents genres musicaux
- Créer de nouvelles variables susceptibles d'apporter des informations supplémentaires au modèle de prédiction.

Les transformations sont réalisées en conservant les données initiales afin de pouvoir comparer différentes approches.



### Analyse de la répartition géographique

Avant de transformer les variables géographiques, on étudie la répartition des individus dans les différentes régions.


Cette analyse permet :
- D'identifier les régions les plus représentées
- De repérer les régions ayant un faible nombre d'individus
- D'évaluer un éventuel déséquilibre dansla distribution des classes à prédire


Cette étape est importante car un déséquilibre entre les régions pourrait rendre la prédiction plus difficile pour les régions les moins représentées.

In [22]:
print("Nombre de personnes :", len(df))
print("Nombre de régions :", df["region"].nunique())
print("Région la plus représentée :", df["region"].value_counts().idxmax())
print("Région la moins représentée :", df["region"].value_counts().idxmin())


Nombre de personnes : 1986
Nombre de régions : 12
Région la plus représentée : Île-de-France
Région la moins représentée : Centre-Val de Loire


# A METTRE DANS NETTOYAGE !!!!!!!!!!!!!!!!!!!!!!!!!!

In [23]:
df_counts = df["region"].value_counts().reset_index(name="count")

In [24]:
px.bar(df_counts, x="region", y="count", title="Nombre d'occurences par région")

### Premier regroupement géographique

Certaines régions disposent d'un nombre d'individus relativement faible. Afin de limiter le nombre de catégories à prédire, on propose de regrouper certaines régions géographiquement proches.


Le regroupement proposé est le suivant : 
| Groupe | Régions associées |
|:-------- |:-------- |
| CENTRE   | Centre-Val de Loire, Pays de la Loire, Bourgogne-Franche-Comté|
| NORD-OUEST | Normandie, Bretagne, Hauts-de-France |
| SUD | Provence-Alpes-Côte d'Azur, Occitanie | 
| Île-de-France | Île-de-France |


Les autres régions conservent leur catégorie d'origine.


Une nouvelle colonne 'region_group' est ainsi créée à partir de la colonne 'region'.


Intérêt : réduire le nombre de classes et potentiellement améliorer la répartition des individus entre les catégories.


Limite : cette simplification entraîne une perte de précision géographique, puisque plusieurs régions distinctes sont désormais associées à une même catégorie.


In [25]:
def groupement(region):
    if region == "Centre-Val de Loire" or region == "Pays de la Loire" or region == "Bourgogne-Franche-Comté":
        return "CENTRE"
    elif region == "Normandie" or region == "Bretagne" or region == "Hauts-de-France":
        return "NORD-OUEST"
    elif region == "Provence-Alpes-Côte d'Azur" or region == "Occitanie":
        return "SUD"
    else:
        return region

In [26]:
df["region_group"] = df["region"].apply(groupement)

In [27]:
df_counts_gr_reg = df["region_group"].value_counts().reset_index(name="count")

In [28]:
px.bar(df_counts_gr_reg, x="region_group", y="count", title="Nombre d'occurences par région")

In [29]:
df["region_group"].value_counts()

region_group
Île-de-France           404
NORD-OUEST              374
SUD                     290
CENTRE                  287
Auvergne-Rhône-Alpes    237
Grand Est               200
Nouvelle-Aquitaine      194
Name: count, dtype: int64

In [30]:
print("Régions originales :", df["region"].nunique())
print("Groupes après transformation :", df["region_group"].nunique())


Régions originales : 12
Groupes après transformation : 7


### Deuxième regroupement géographique : approche cardinale

On propose une seconde méthode de regroupement, plus structurée, consistant à répartir les régions françaises en grandes zones géographiques.


Cinq catégories sont définies :

| Groupe | Régions associées |
|:-------- |:-------- |
| NORD   | Normandie, Hauts-de-France |
| SUD | Provence-Alpes-Côte d'Azur, Occitanie |
| EST | Grand Est, Bourgogne-Franche-Comté |
| OUEST | Bretagne, Pays de la Loire, Nouvelle-Aquitaine |
| CENTRE | Centre-Val de Loire, Auvergne-Rhône-Alpes |
| Île-de-France | Île-de-France |


Une nouvelle colonne 'region_group_cardinal' est créée.


Cette approche permet de tester une autre granularité géographique et de comparer les résultats avec le premier regroupement.


L'objectif est de déterminer si une classification géographique plus générale permet d'obtenir des catégories mieux réparties, tout en conservant une information géographique pertinente.


Les deux méthodes sont conservées afin de pouvoir comparer leurs performances lors de la phase de modélisation.


In [31]:
def groupement_cardinal(region):
    if region == "Centre-Val de Loire" or region == "Auvergne-Rhône-Alpes":
        return "CENTRE"
    elif region == "Normandie" or region == "Hauts-de-France":
        return "NORD"
    elif region == "Provence-Alpes-Côte d'Azur" or region == "Occitanie":
        return "SUD"
    elif region == "Bretagne" or region == "Pays de la Loire" or region == "Nouvelle-Aquitaine":
        return "OUEST"
    elif region == "Grand Est" or region == "Bourgogne-Franche-Comté":
        return "EST"
    else:
        return region

In [32]:
df["region_group_cardinal"] = df["region"].apply(groupement_cardinal)
df_counts_gr_reg_car = df["region_group_cardinal"].value_counts().reset_index(name="count")
px.bar(df_counts_gr_reg_car, x="region_group_cardinal", y="count", title="Nombre d'occurences par région")

# Analyse de la fréquence des genres musicaux
Les variables musicales sont représentées sous forme binaire : 
- 1 : le genre musical a été sélectionné par l'individu
- 0 : le genre musical n'a pas été sélectionné

Nous étudions la répartition de ces deux valeurs pour chaque genre musical à l'aide de diagrammes circulaires.

Cette visualisation permet d'identifier :
- Les genres musicaux les plus populaires
- Les genres les moins représentés
- Les différences de fréquence de sélection entre les différents genres

Cette analyse descriptive permet de mieux comprendre la composition du dataset et les caractéristiques des préférences musicales des individus. 

In [33]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import math
from IPython.display import display

# Récupérer les colonnes de type int64
columns = [column for column in df.columns if df[column].dtype == "int64"]

# Définir la grille
n_cols = 3
n_rows = math.ceil(len(columns) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "domain"} for _ in range(n_cols)]
           for _ in range(n_rows)],
    subplot_titles=[f"Répartition de {column}" for column in columns]
)

for i, column in enumerate(columns):
    df_prop = (
        df[column]
        .value_counts(normalize=True)
        .reset_index(name="proportion")
        .sort_values(by=column, ascending=True)
        .replace({1: "selectionné", 0: "non-selectionné"})
    )

    fig.add_trace(
        go.Pie(
            labels=df_prop[column],
            values=df_prop["proportion"],
            textinfo="percent+label",
            name=column
        ),
        row=i // n_cols + 1,
        col=i % n_cols + 1
    )

fig.update_layout(
    title_text="Répartition des variables numériques",
    height=400 * n_rows,
    width=1000,
    showlegend=False
)

fig.show()

# Nombre de styles écoutés

### Création de la variable 'total_genres'

Afin d'enrichir le dataset, on crée une nouvelle variable appelée 'total_genres'.

Cette variable correspond au nombre total de genres musicaux sélectionnés par chaque individu.

Puisque les genres sont représentés par des variables binaires (0 ou 1), il suffit de calculer la somme des colonnes musicales pour chaque individu.

Cette nouvelle variable permet de mesurer la diversité des préférences musicales de chaque personne.

Elle pourra notamment être utilisée pour étudier si le nombre de genres sélectionnés présente des différences selon la région géographique.

In [ ]:
#df["total_genres"] = df.iloc[:,0:13].sum(axis=1)

In [ ]:
df["total_genres"] = df[colonnes_genres].sum(axis=1)

## Comparaison du nombre de genres selon la région

Nous étudions maintenant le nombre moyen de genres musicaux sélectionnés par individu dans chaque région.

Pour cela, nous calculons :
- La moyenne de 'total_genres' pour chaque région.
- Le nombre d'individus associés à chaque région.

Cette comparaison permet de rechercher d'éventuelles différences dans la diversité des préférences musicales selon la localisation géographique.

Il est toutefois important de rappeler qu'une différence observée entre les moyennes ne signifie pas nécessairement que cette variable sera pertinente pour prédire la région.

Cette analyse constitue donc une première exploration des relations possibles entre les variables musicales et géographiques.

In [35]:
moyenne_genres = df.groupby("region")["total_genres"].agg(["mean", "count"]).sort_values("mean", ascending=False)
display(moyenne_genres.round(2))
fig = px.bar(moyenne_genres.reset_index(), x="region", y="mean", title="Nombre moyen de genres sélectionnés selon la région", labels={"region": "Région", "mean": "Nombre moyen de genres"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()


,mean,count
region,,
Bourgogne-Franche-Comté,3.67,84
Pays de la Loire,3.64,125
Centre-Val de Loire,3.62,78
Nouvelle-Aquitaine,3.60,194
Grand Est,3.56,200
Occitanie,3.51,161
Provence-Alpes-Côte d'Azur,3.47,129
Normandie,3.42,89
Bretagne,3.38,97


## Vérifications et export

Avant l'exportation, plusieurs vérifications sont effectuées afin de s'assurer de la cohérence des transformations :

- Vérification de la présence des colonnes géographiques initiales et transformées.
- Vérification de l'absence de valeurs manquantes dans les colonnes 'region' et 'region_group'.
- Vérification que le dataset contient bien des observations.
- Vérification de la cohérence de la variable 'total_genres' avec la somme des variables musicales.

Le dataset transformé est ensuite exporté dans un nouveau fichier CSV nommé 'dataset_musique_region_transforme.csv'.

Le fichier initial n'est pas écrasé, ce qui permet de conserver les données originales et de comparer les différentes étapes du traitement.

In [36]:
assert "region" in df.columns and "region_group" in df.columns
assert df["region"].notna().all() and df["region_group"].notna().all()
assert len(df) > 0
assert df["total_genres"].eq(df[colonnes_genres].sum(axis=1)).all()

chemin_sortie = Path(filepath).with_name("dataset_musique_region_transforme.csv")

df.to_csv(chemin_sortie, index=False)
print("Fichier transformé enregistré :", chemin_sortie)
print("Dimensions :", df.shape)


Fichier transformé enregistré : ..\Dataset\dataset_musique_region_transforme.csv
Dimensions : (1986, 17)


# Bilan des transformations

Au cours de cette étape, plusieurs transformations et analyses ont été réalisées afin de préparer les données pour la future phase de modélisation.

Les principales opérations effectuées sont :

1. **Analyse de la répartition géographique** : identification des régions les plus et les moins représentées.
2. **Premier regroupement géographique** : création de la variable 'region_group'.
3. **Deuxième regroupement géographique** : création de la variable 'region_group_cardinal', basée sur cinq grandes zones.
4. **Analyse des préférences musicales** : étude de la fréquence de sélection de chaque genre.
5. **Création d'une variable supplémentaire** : ajout de 'total_genres', représentant le nombre de genres sélectionnés par individu.
6. **Vérification et export** : contrôle de la cohérence des nouvelles variables et sauvegarde du dataset transformé.

Le dataset conserve les variables initiales et intègre de nouvelles variables permettant d'explorer différentes représentations de l'information géographique et musicale.

Ces transformations permettront de comparer plusieurs configurations de données lors de la phase de modélisation et d'évaluer leur influence sur les performances de prédiction géographique.